
# Gold Run Evidence

This notebook is run manually after the Gold notebooks (`01_gold_dim_province`, `02_gold_dim_year` and `03_gold_fact_province_year`) have completed. It is not part of the pipeline and does not write any data. It reads the Gold tables, `gold_run_log` and `dq_results` and reports what those runs produced.

| Output | Purpose |
| --- | --- |
| Row counts | Confirms that `dim_province` holds 100 rows, `dim_year` holds 6 rows and `fact_province_year` holds 600 rows |
| Latest run per notebook | Reports the final status of each notebook with its check, BLOCK failure and REVIEW flag counts |
| Check summary | Summarises PASS and FAIL results by severity |
| All checks | Lists every check from those runs, with failures first, so that any REVIEW flag is disclosed |
| Sample fact rows | Shows representative rows from `fact_province_year`, with province names joined from `dim_province` |
| Caveat fields | Shows, for each caveat carried over from Silver, the provinces and years it applies to |

The final step prints markdown for `docs/gold-run-evidence.md`.

In [0]:
from pyspark.sql import functions as F

CATALOG = "tuklas_dev"          # replace with the catalog widget once the notebook-controls change is merged
G = f"`{CATALOG}`.`04_gold`"
Q = f"`{CATALOG}`.`06_data_quality`"

def md_table(df, max_rows=1000):
    cols = df.columns
    rows = df.limit(max_rows).collect()
    line = lambda vals: "| " + " | ".join("" if v is None else str(v).replace("|", "/").replace("\n", " ") for v in vals) + " |"
    return "\n".join([line(cols), line(["---"] * len(cols))] + [line(r) for r in rows])

# 1. Row counts
counts = spark.createDataFrame(
    [(t, spark.table(f"{G}.{t}").count(), e) for t, e in [("dim_province", 100), ("dim_year", 6), ("fact_province_year", 600)]],
    "table_name STRING, rows BIGINT, expected BIGINT").withColumn("ok", F.col("rows") == F.col("expected"))

# 2. Latest run per notebook, with its final status
latest = spark.sql(f"""
SELECT notebook_name, run_id, status, tables_written, rows_written, dq_checks_run, dq_block_failures, dq_review_flags, notes
FROM (SELECT *, ROW_NUMBER() OVER (PARTITION BY notebook_name ORDER BY started_at DESC) AS rn FROM {G}.gold_run_log)
WHERE rn = 1 ORDER BY notebook_name""")
run_ids = [r["run_id"] for r in latest.collect()]

# 3. Every check from those runs, failures first, so REVIEW flags are visible
checks = (spark.table(f"{Q}.dq_results").filter(F.col("run_id").isin(run_ids))
          .select("notebook_name", "check_id", "severity", "status", "failed_rows", "observed")
          .orderBy(F.when(F.col("status") == "FAIL", 0).otherwise(1), "notebook_name", "check_id"))
summary = (spark.table(f"{Q}.dq_results").filter(F.col("run_id").isin(run_ids))
           .groupBy("severity", "status").count().orderBy("severity", "status"))

# 4. Sample rows, with names joined from the dimension (the fact keeps the code only)
fact = (spark.table(f"{G}.fact_province_year")
        .join(spark.table(f"{G}.dim_province").select("province_psgc", "province_name"), "province_psgc"))
sample_fact = (fact.select("province_psgc", "province_name", "year", "total_travelers", "traveler_value_status",
                           "gdp_total_current", "population_total", "comparable_to_2019")
               .orderBy("province_psgc", "year").limit(8))

# 5. Caveat fields carried over from Silver: provinces and years they apply to
def caveat_rows(label, condition, example=F.lit(None).cast("string")):
    return (fact.filter(condition).groupBy("province_name")
            .agg(F.min("year").alias("first_year"), F.max("year").alias("last_year"), F.min(example).alias("example"))
            .withColumn("caveat", F.lit(label)))

caveats = (
    caveat_rows("comparable_to_2019 = false", ~F.col("comparable_to_2019"))
    .unionByName(caveat_rows("foreign_includes_overseas_filipinos = true", F.col("foreign_includes_overseas_filipinos")))
    .unionByName(caveat_rows("derived_values filled", F.col("derived_values").isNotNull(), F.col("derived_values")))
    .unionByName(caveat_rows("per_capita_recomputed = true", F.col("per_capita_recomputed")))
    .select("caveat", "province_name", "first_year", "last_year", "example")
    .orderBy("caveat", "province_name"))

sections = [("Row counts", counts), ("Latest run per notebook", latest), ("Check summary", summary),
            ("All checks (failures first)", checks), ("Sample fact rows", sample_fact), ("Caveat fields", caveats)]

for title, df in sections:
    print(f"\n### {title}\n"); display(df)

doc = "\n\n".join(f"### {title}\n\n{md_table(df)}" for title, df in sections)
print("\n===== COPY BELOW INTO docs/gold-run-evidence.md =====\n"); print(doc)